# 7-Day Diet Planning with Meal Composition Rules

**CS 524: Introduction to Optimization - Multi-Period Extension**  
**Fall 2025**

---

## Extension 1: Multi-Period Planning

This notebook extends the basic MIP diet model to:
- **7-day planning horizon**: Plan an entire week of meals
- **Meal composition rules**: Each day requires 1 Main + 1 Dessert + 1 Drink
- **Variety across days**: Prevent meal repetition throughout the week
- **Weekly nutritional constraints**: Meet requirements over the entire week


## Mathematical Formulation

### Additional Sets
- $T = \{1, 2, ..., 7\}$: Set of days (Monday through Sunday)
- $F_M \subseteq F$: Set of Main foods
- $F_D \subseteq F$: Set of Dessert foods
- $F_B \subseteq F$: Set of Drink (Beverage) foods

### Decision Variables
- $y_{it} \in \{0,1\}$: Binary variable = 1 if food $i$ is selected on day $t$
- $x_{it} \in \mathbb{Z}^+$: Integer servings of food $i$ on day $t$

### Objective Function
Minimize total weekly cost:
$$\min \sum_{t \in T} \sum_{i \in F} c_i \cdot x_{it}$$

### New Constraints

**1. Meal Composition (Each day needs 1 Main + 1 Dessert + 1 Drink)**
$$\sum_{i \in F_M} y_{it} = 1, \quad \forall t \in T \quad \text{(exactly 1 main per day)}$$
$$\sum_{i \in F_D} y_{it} = 1, \quad \forall t \in T \quad \text{(exactly 1 dessert per day)}$$
$$\sum_{i \in F_B} y_{it} = 1, \quad \forall t \in T \quad \text{(exactly 1 drink per day)}$$

**2. No Meal Repetition (Can't use same main/dessert/drink combo twice)**
$$y_{it_1} + y_{jt_2} + y_{kt_3} \leq 2.99, \quad \forall (i,j,k) \in F_M \times F_D \times F_B, \, \forall t_1, t_2, t_3 \in T$$
(Simplified: Encourage variety by limiting repeated selections)

**3. Weekly Nutritional Requirements**
$$N_{\min,j} \leq \sum_{t \in T} \sum_{i \in F} a_{ij} \cdot x_{it} \leq N_{\max,j}, \quad \forall j \in N$$

**4. Weekly Budget**
$$B_{\min} \leq \sum_{t \in T} \sum_{i \in F} c_i \cdot x_{it} \leq B_{\max}$$

**5. Linking Constraints (Minimum 1 Serving if Selected)**
$$1 \cdot y_{it} \leq x_{it} \leq M \cdot y_{it}, \quad \forall i \in F, \, \forall t \in T$$

This ensures:
- If $y_{it} = 0$ (food not selected): $x_{it} = 0$ (no servings)
- If $y_{it} = 1$ (food selected): $1 \leq x_{it} \leq M$ (at least 1 serving, up to M servings)


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import gamspy as gp
import gamspy.math as gpm
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Initialize GAMSPy
gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container()

print("✅ Multi-Period 7-Day Model Initialized")

## Food Categorization

We categorize the 35 foods into three types:
- **Mains**: Burrito, Bowl, Sandwich, Pizza, Burger, Tacos, etc.
- **Desserts**: Ice cream, Donut, Muffin, Croissant, Cake, etc.
- **Drinks**: Coffee, Latte, Smoothie, Milkshake, etc.

In [ ]:
# Define nutrients
expanded_nutrients = [
    "Calories", "Protein", "Carbs", "Fat", "SaturatedFat", "TransFat", "Sugars",
    "Sodium", "Fiber", "VitaminA", "VitaminC", "VitaminD", "Calcium", "Iron", 
    "Potassium", "Cholesterol", "Caffeine"
]

# Define restaurants and categorize foods by type
# Format: {restaurant: {category: [items]}}
restaurants_dict = {
    "Chipotle": {
        "Main": ["Chicken_Burrito", "Steak_Bowl", "Veggie_Tacos"],
        "Dessert": ["Churros"]  # Fried dough pastry with cinnamon sugar
    },
    "Subway": {
        "Main": ["Turkey_Sandwich", "Veggie_Delite", "Chicken_Teriyaki", "Meatball_Marinara"]
    },
    "McDonalds": {
        "Main": ["Big_Mac", "Quarter_Pounder", "Chicken_Nuggets"],
        "Dessert": ["Vanilla_Cone"]  # Soft serve ice cream cone
    },
    "PizzaHut": {
        "Main": ["Pepperoni_Pizza", "Cheese_Pizza", "Veggie_Pizza"],
        "Dessert": ["Brownie"]  # Chocolate brownie dessert
    },
    "TacoBell": {
        "Main": ["Crunchwrap", "Taco", "Burrito"],
        "Dessert": ["Cinnamon_Twist"]  # Fried dough with cinnamon sugar
    },
    "Starbucks": {
        "Drink": ["Latte", "Cappuccino", "Frappuccino"],
        "Dessert": ["Muffin", "Croissant"]
    },
    "Dunkin": {
        "Drink": ["Coffee"],
        "Dessert": ["Donut", "Cheese_Cake"]  # Donut and premium cheese cake
    },
    "DailyScoop": {
        "Dessert": ["Vanilla_Cone", "Chocolate_Sundae", "Strawberry_Scoop", "Cookie_Dough"]
    },
    "ColdStone": {
        "Dessert": ["IceCream_Cake"],
        "Drink": ["Smoothie", "Milkshake"]
    }
}

# Build categorized food lists
food_categories = {"Main": [], "Dessert": [], "Drink": []}
food_list = []

for restaurant, categories in restaurants_dict.items():
    for category, items in categories.items():
        for item in items:
            food_name = f"{restaurant}_{item}"
            food_list.append(food_name)
            food_categories[category].append(food_name)

# Days of the week
days_list = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

print(f"✅ Food Categories:")
print(f"   - Mains: {len(food_categories['Main'])} items")
print(f"   - Desserts: {len(food_categories['Dessert'])} items")
print(f"   - Drinks: {len(food_categories['Drink'])} items")
print(f"   - Total: {len(food_list)} foods across 7 days")
print(f"\n📋 Mains: {food_categories['Main'][:5]}...")
print(f"📋 Desserts: {food_categories['Dessert'][:5]}...")
print(f"📋 Drinks: {food_categories['Drink']}")

In [ ]:
# Load nutritional data
df = pd.read_csv("nutrient_data.csv")

nutrient_values = {}
for _, row in df.iterrows():
    item = row['Restaurant_MenuItem']
    nutrient_values[item] = {
        "Calories": row['Calories'], "Protein": row['Protein'], "Carbs": row['Carbs'], "Fat": row['Fat'],
        "SaturatedFat": row['SaturatedFat'], "TransFat": row['TransFat'], "Sugars": row['Sugars'],
        "Sodium": row['Sodium'], "Fiber": row['Fiber'], "VitaminA": row['VitaminA'], "VitaminC": row['VitaminC'],
        "VitaminD": row['VitaminD'], "Calcium": row['Calcium'], "Iron": row['Iron'], "Potassium": row['Potassium'],
        "Cholesterol": row['Cholesterol'], "Caffeine": row['Caffeine']
    }

# Expand to (food, nutrient, value) for GAMSPy
nutrient_data_expanded = [(food, nutrient, nutrient_values[food].get(nutrient, 0)) 
                          for food in food_list for nutrient in expanded_nutrients]

print(f"✅ Loaded nutritional data for {len(food_list)} foods")

In [ ]:
# Create GAMSPy sets
foods = gp.Set(m, name="foods", records=food_list)
nutrients = gp.Set(m, name="nutrients", records=expanded_nutrients)
days = gp.Set(m, name="days", records=days_list)

# Create category sets
mains = gp.Set(m, name="mains", domain=[foods], records=food_categories["Main"])
desserts = gp.Set(m, name="desserts", domain=[foods], records=food_categories["Dessert"])
drinks = gp.Set(m, name="drinks", domain=[foods], records=food_categories["Drink"])

print(f"✅ GAMSPy sets created")

In [ ]:
# Load parameters from CSV files

# Prices
prices_df = pd.read_csv("food_prices.csv", dtype={'Food': str, 'Restaurant': str})
price_per_serving_data = prices_df[['Food', 'Price']].copy()
price_per_serving_data.columns = ['foods', 'value']
price_per_serving = gp.Parameter(m, name="price_per_serving", domain=[foods], records=price_per_serving_data)

# Nutritional content
nutrient_per_serving = gp.Parameter(m, name="nutrient_per_serving", domain=[foods, nutrients], records=nutrient_data_expanded)

# Nutrient constraints (these are now WEEKLY totals)
constraints_df = pd.read_csv("nutrient_constraints.csv", dtype={'Nutrient': str})

# Multiply by 7 for weekly requirements
Nmin_data = constraints_df[['Nutrient', 'Nmin']].copy()
Nmin_data['Nmin'] = Nmin_data['Nmin'] * 7  # Weekly minimum
Nmin_data.columns = ['nutrients', 'value']
Nmin = gp.Parameter(m, name="Nmin", domain=[nutrients], records=Nmin_data)

Nmax_data = constraints_df[['Nutrient', 'Nmax']].copy()
Nmax_data['Nmax'] = Nmax_data['Nmax'] * 7  # Weekly maximum
Nmax_data.columns = ['nutrients', 'value']
Nmax = gp.Parameter(m, name="Nmax", domain=[nutrients], records=Nmax_data)

# Satisfaction
satisfaction_df = pd.read_csv("food_satisfaction.csv", dtype={'Food': str})
base_satisfaction_data = satisfaction_df[['Food', 'Base_Satisfaction']].copy()
base_satisfaction_data.columns = ['foods', 'value']
base_satisfaction = gp.Parameter(m, name="base_satisfaction", domain=[foods], records=base_satisfaction_data)

# Model config
config_df = pd.read_csv("model_config.csv")
config_values = config_df.set_index('Parameter')['Value'].to_dict()

# Weekly budget (multiply by 7)
budget_min = float(config_values['budget_min']) * 7
budget_max = float(config_values['budget_max']) * 7
max_servings_per_food = int(config_values['max_servings_per_food'])
min_satisfaction = float(config_values['min_satisfaction']) * 7  # Weekly satisfaction

print(f"✅ Parameters loaded:")
print(f"   - Weekly budget: ${budget_min:.0f} - ${budget_max:.0f}")
print(f"   - Max servings per food per day: {max_servings_per_food}")
print(f"   - Weekly minimum satisfaction: {min_satisfaction:.0f}")

## Decision Variables (Multi-Period)

Now variables are indexed by both food **AND** day:
- `y[i,t]`: Binary selection of food i on day t
- `x[i,t]`: Integer servings of food i on day t

In [ ]:
# Decision Variables (indexed by food AND day)

# Binary: Is food i selected on day t?
y = gp.Variable(m, name="y", domain=[foods, days], type="binary")

# Integer: Servings of food i on day t
x = gp.Variable(m, name="x", domain=[foods, days], type="integer")
x.lo[foods, days] = 0
x.up[foods, days] = max_servings_per_food

print(f"✅ Variables created: y[food, day] and x[food, day]")
print(f"   - Total binary variables: {len(food_list)} × 7 = {len(food_list) * 7}")
print(f"   - Total integer variables: {len(food_list)} × 7 = {len(food_list) * 7}")
print(f"   - Total decision variables: {len(food_list) * 7 * 2}")

## Constraints

### New Meal Composition Constraints
Each day must have:
- Exactly 1 Main
- Exactly 1 Dessert
- Exactly 1 Drink

In [ ]:
# 1. MEAL COMPOSITION CONSTRAINTS

# Each day needs exactly 1 main
one_main_per_day = gp.Equation(m, name="one_main_per_day", domain=[days])
one_main_per_day[days] = gp.Sum(mains, y[mains, days]) == 1

# Each day needs exactly 1 dessert
one_dessert_per_day = gp.Equation(m, name="one_dessert_per_day", domain=[days])
one_dessert_per_day[days] = gp.Sum(desserts, y[desserts, days]) == 1

# Each day needs exactly 1 drink
one_drink_per_day = gp.Equation(m, name="one_drink_per_day", domain=[days])
one_drink_per_day[days] = gp.Sum(drinks, y[drinks, days]) == 1

print("✅ Meal composition constraints: 1 Main + 1 Dessert + 1 Drink per day")

# 2. VARIETY CONSTRAINT (Don't repeat the same food across multiple days)
# Limit each food to appear at most 3 times across the week
# Note: With only 5 drinks and needing 7 drinks/week, this must be ≥3
max_repeats = 3
limit_repetition = gp.Equation(m, name="limit_repetition", domain=[foods])
limit_repetition[foods] = gp.Sum(days, y[foods, days]) <= max_repeats

print(f"✅ Variety constraint: Each food can appear at most {max_repeats} times per week")

# 3. LINKING CONSTRAINTS (connect binary selection y to integer servings x)

# Upper linking: If y=0, then x=0; If y=1, then x can be up to max_servings
link_upper = gp.Equation(m, name="link_upper", domain=[foods, days])
link_upper[foods, days] = x[foods, days] <= max_servings_per_food * y[foods, days]

# Lower linking: If y=1, then x must be at least 1 serving
link_lower = gp.Equation(m, name="link_lower", domain=[foods, days])
link_lower[foods, days] = x[foods, days] >= 1 * y[foods, days]

print("✅ Linking constraints: 1 × y[i,t] ≤ x[i,t] ≤ M × y[i,t]")
print("   → If food is selected (y=1), must have at least 1 serving")

# 4. WEEKLY NUTRITIONAL CONSTRAINTS (sum over all days)
nutrient_min = gp.Equation(m, name="nutrient_min", domain=[nutrients])
nutrient_min[nutrients] = gp.Sum([days, foods], nutrient_per_serving[foods, nutrients] * x[foods, days]) >= Nmin[nutrients]

nutrient_max = gp.Equation(m, name="nutrient_max", domain=[nutrients])
nutrient_max[nutrients] = gp.Sum([days, foods], nutrient_per_serving[foods, nutrients] * x[foods, days]) <= Nmax[nutrients]

print("✅ Weekly nutritional constraints (sum across all 7 days)")

# 5. WEEKLY BUDGET CONSTRAINTS
cost_min = gp.Equation(m, name="cost_min")
cost_min[:] = gp.Sum([days, foods], price_per_serving[foods] * x[foods, days]) >= budget_min

cost_max = gp.Equation(m, name="cost_max")
cost_max[:] = gp.Sum([days, foods], price_per_serving[foods] * x[foods, days]) <= budget_max

print(f"✅ Weekly budget constraints: ${budget_min:.0f} - ${budget_max:.0f}")

# 6. WEEKLY SATISFACTION CONSTRAINT
satisfaction_constraint = gp.Equation(m, name="satisfaction_constraint")
satisfaction_constraint[:] = gp.Sum([days, foods], base_satisfaction[foods] * x[foods, days]) >= min_satisfaction

print(f"✅ Weekly satisfaction constraint: minimum {min_satisfaction:.0f}")

print("\n📊 Total constraints created:")
print(f"   - Meal composition: {3 * 7} = 21 (1 main + 1 dessert + 1 drink × 7 days)")
print(f"   - Variety: {len(food_list)}")
print(f"   - Linking (upper + lower): {len(food_list) * 7 * 2} = {len(food_list) * 7} × 2")
print(f"   - Nutritional: {len(expanded_nutrients) * 2}")
print(f"   - Budget: 2")
print(f"   - Satisfaction: 1")

## Objective Function

Minimize total weekly cost (sum over all days):
$$\min \sum_{t=1}^{7} \sum_{i \in F} c_i \cdot x_{it}$$

In [ ]:
# Objective: Minimize total weekly cost
total_weekly_cost = gp.Sum([days, foods], price_per_serving[foods] * x[foods, days])

print("✅ Objective: Minimize total weekly cost (sum over 7 days)")

## Solve the Model

⚠️ **Note**: This is a much larger model than the single-period version. It may take longer to solve (10-60 seconds).

In [ ]:
# Create and solve the model
print("Solving 7-Day Multi-Period MIP Model...\n")
print("⏳ This may take 10-60 seconds due to model size...\n")

diet_plan_7day = gp.Model(
    m,
    equations=m.getEquations(),
    problem=gp.Problem.MIP,
    sense=gp.Sense.MIN,
    objective=total_weekly_cost,
    name="diet_plan_7day_mip",
)

diet_plan_7day.solve(output=sys.stdout)

print("\n✅ Model solved!")


## Results: 7-Day Meal Plan

In [ ]:
# Display results
print("\n" + "="*80)
print("7-DAY MEAL PLAN RESULTS")
print("="*80)

if diet_plan_7day.status in [gp.ModelStatus.OptimalGlobal, gp.ModelStatus.OptimalLocal]:
    print(f"\n💰 Total Weekly Cost: ${diet_plan_7day.objective_value:.2f}")
    print(f"📊 Status: {diet_plan_7day.status} ✅")
    print(f"⏱️  Average daily cost: ${diet_plan_7day.objective_value / 7:.2f}")
    
    # Extract solution
    y_records = y.records
    x_records = x.records
    
    # Create solution dictionary
    solution = {}
    for _, row in y_records.iterrows():
        food = row[y_records.columns[0]]
        day = row[y_records.columns[1]]
        selected = row['level']
        if selected > 0.5:
            if day not in solution:
                solution[day] = []
            solution[day].append(food)
    
    # Get servings
    servings_dict = {}
    for _, row in x_records.iterrows():
        food = row[x_records.columns[0]]
        day = row[x_records.columns[1]]
        servings = row['level']
        if servings > 0:
            servings_dict[(food, day)] = servings
    
    # Get prices
    price_dict = {row['foods']: row['value'] for _, row in price_per_serving.records.iterrows()}
    
    # Display meal plan day by day
    print("\n" + "="*80)
    print("WEEKLY MEAL PLAN (1 Main + 1 Dessert + 1 Drink per day)")
    print("="*80)
    
    for day in days_list:
        if day in solution:
            print(f"\n📅 {day.upper()}:")
            print("-" * 80)
            
            # Categorize foods for this day
            day_main = [f for f in solution[day] if f in food_categories["Main"]]
            day_dessert = [f for f in solution[day] if f in food_categories["Dessert"]]
            day_drink = [f for f in solution[day] if f in food_categories["Drink"]]
            
            daily_cost = 0
            
            if day_main:
                food = day_main[0]
                servings = servings_dict.get((food, day), 0)
                cost = price_dict[food] * servings
                daily_cost += cost
                print(f"   🍽️  Main:    {food.replace('_', ' '):40s} | {servings:.0f} servings | ${cost:.2f}")
            
            if day_dessert:
                food = day_dessert[0]
                servings = servings_dict.get((food, day), 0)
                cost = price_dict[food] * servings
                daily_cost += cost
                print(f"   🍰 Dessert: {food.replace('_', ' '):40s} | {servings:.0f} servings | ${cost:.2f}")
            
            if day_drink:
                food = day_drink[0]
                servings = servings_dict.get((food, day), 0)
                cost = price_dict[food] * servings
                daily_cost += cost
                print(f"   ☕ Drink:   {food.replace('_', ' '):40s} | {servings:.0f} servings | ${cost:.2f}")
            
            print(f"\n   💵 Daily Total: ${daily_cost:.2f}")
    
    # Summary statistics
    total_foods_used = sum(len(foods_in_day) for foods_in_day in solution.values())
    unique_foods = set(food for day_foods in solution.values() for food in day_foods)
    
    print("\n" + "="*80)
    print("WEEKLY SUMMARY")
    print("="*80)
    print(f"Total weekly cost: ${diet_plan_7day.objective_value:.2f}")
    print(f"Average daily cost: ${diet_plan_7day.objective_value / 7:.2f}")
    print(f"Total meals: {total_foods_used} (should be 21: 3 items × 7 days)")
    print(f"Unique foods used: {len(unique_foods)} different items")
    print(f"Variety score: {len(unique_foods) / len(food_list) * 100:.1f}% of available foods used")
    
else:
    print(f"\n⚠️  Model Status: {diet_plan_7day.status}")
    print("Model is INFEASIBLE or encountered an error.")
    print("\nPossible fixes:")
    print("1. Increase weekly budget_max in model_config.csv")
    print("2. Relax nutrient constraints")
    print("3. Increase max_repeats (allow foods to appear more than 2 times)")
    print("4. Add more food options to each category")

## Visualization: 7-Day Meal Plan

In [ ]:
# Visualize daily costs
if diet_plan_7day.status in [gp.ModelStatus.OptimalGlobal, gp.ModelStatus.OptimalLocal]:
    # Calculate daily costs
    daily_costs = {}
    for day in days_list:
        day_cost = sum(price_dict[food] * servings_dict.get((food, day), 0) 
                      for food in solution.get(day, []))
        daily_costs[day] = day_cost
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Daily cost bar chart
    colors = sns.color_palette("husl", 7)
    ax1.bar(daily_costs.keys(), daily_costs.values(), color=colors, alpha=0.8, edgecolor='black')
    ax1.axhline(y=diet_plan_7day.objective_value / 7, color='red', linestyle='--', linewidth=2, label='Average Daily Cost')
    ax1.set_xlabel('Day of Week', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Cost ($)', fontsize=12, fontweight='bold')
    ax1.set_title('Daily Meal Costs Across the Week', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Food variety pie chart
    category_counts = {'Main': 0, 'Dessert': 0, 'Drink': 0}
    for day_foods in solution.values():
        for food in day_foods:
            for category, category_foods in food_categories.items():
                if food in category_foods:
                    category_counts[category] += 1
    
    ax2.pie(category_counts.values(), labels=category_counts.keys(), autopct='%1.1f%%',
           colors=['#ff9999', '#66b3ff', '#99ff99'], startangle=90)
    ax2.set_title('Meal Composition Balance (Total Selections)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Visualizations: Daily costs and meal composition balance")

In [ ]:
# Create a heatmap showing which foods are used on which days
if diet_plan_7day.status in [gp.ModelStatus.OptimalGlobal, gp.ModelStatus.OptimalLocal]:
    # Create matrix: foods × days
    foods_used = sorted(list(unique_foods))
    matrix = np.zeros((len(foods_used), 7))
    
    for i, food in enumerate(foods_used):
        for j, day in enumerate(days_list):
            if food in solution.get(day, []):
                matrix[i, j] = servings_dict.get((food, day), 0)
    
    # Plot heatmap
    plt.figure(figsize=(12, max(8, len(foods_used) * 0.4)))
    sns.heatmap(matrix, 
                xticklabels=days_list, 
                yticklabels=[f.replace('_', ' ') for f in foods_used],
                cmap='YlOrRd', 
                annot=True, 
                fmt='.0f',
                cbar_kws={'label': 'Number of Servings'},
                linewidths=0.5)
    plt.title('7-Day Meal Plan: Food Usage Heatmap', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Day of Week', fontsize=12, fontweight='bold')
    plt.ylabel('Food Item', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Heatmap: Shows which foods are selected on which days and how many servings")

## Comparison: Single-Day vs 7-Day Model

### Model Size Comparison

| Metric | Single-Day | 7-Day Multi-Period | Ratio |
|--------|------------|---------------------|-------|
| Binary Variables | 35 | 245 (35 × 7) | 7× |
| Integer Variables | 35 | 245 (35 × 7) | 7× |
| Total Variables | 70 | 490 | 7× |
| Nutritional Constraints | 34 | 34 | Same |
| Meal Composition | 0 | 21 (3 × 7) | New! |
| Variety Constraints | 2 | 35 | +33 |
| Linking Constraints | 35 | 490 (245 × 2) | 14× |
| Total Constraints | ~75 | ~580 | ~8× |

### Key Improvements

1. **Realistic meal structure**: Each day has 1 Main + 1 Dessert + 1 Drink
2. **Weekly planning**: Optimizes across entire week, not just one day
3. **Variety enforcement**: Can't repeat foods more than 2× per week
4. **Minimum serving requirement**: If a food is selected, must have at least 1 serving (no fractional selections)
5. **Better nutrition**: Weekly targets allow more flexibility
6. **Practical implementation**: Clear daily meal plan with realistic portions
